# AMLC26 entity resolution: Kaggle runner

Run the cells top to bottom. Every heavy step runs as a subprocess and logs to `$BER_WORK/logs/`.
`block.py` / `features.py` **resume** from existing part files, so after changing blocking or
feature code, call `fresh(...)` on the affected folders before rerunning (otherwise stale parts are reused).

Settings: Internet ON. Accelerator is optional: the pipeline is CPU-bound (polars/rapidfuzz); only
LightGBM can use a GPU, and only if cell 3 reports that it works.

In [ ]:
# 1. Clone / fast-forward the repo (never deletes a checkout with local changes)
import os, subprocess, glob, shutil, time, json
from pathlib import Path

REPO = Path('/kaggle/working/ml-challenge')
URL = 'https://github.com/hazelmayank/ml-challenge.git'
BRANCH = 'main'

if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, URL, str(REPO)], check=True)
elif (REPO / '.git').is_dir():
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Kaggle checkout has local changes; review them before pulling.')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    raise RuntimeError(f'{REPO} exists but is not a git checkout; inspect it first.')
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('commit', COMMIT)

In [ ]:
# 2. Paths. BER_DATA must contain train/ and test/ with the TSVs; auto-detected under /kaggle/input.
hits = glob.glob('/kaggle/input/**/train/train_source1.tsv', recursive=True)
assert hits, 'dataset not found under /kaggle/input: attach the dataset'
DATA = str(Path(hits[0]).parents[1])
SRC = REPO / 'code' / 'business_entity_resolution' / 'src'
VALIDATOR = REPO / 'student_resource' / 'utils' / 'validate_submission.py'

def set_paths(work, output='/kaggle/working/output'):
    os.environ.update(BER_DATA=DATA, BER_WORK=work, BER_OUTPUT=output)
    for p in (work, output, f'{work}/logs'):
        Path(p).mkdir(parents=True, exist_ok=True)

set_paths('/kaggle/working/work')
print('data', DATA, os.listdir(DATA))

In [ ]:
# 3. Install pinned deps, show resources, and test whether LightGBM can use the GPU here.
subprocess.run(['pip', 'install', '-q', '-r', str(REPO / 'code/business_entity_resolution/requirements.txt')], check=True)
subprocess.run('nproc; free -g; df -h /kaggle/working /tmp; nvidia-smi -L || true', shell=True)

gpu_test = '''
import numpy as np, lightgbm as lgb, sys, time
X = np.random.rand(200_000, 40).astype(np.float32); y = (X[:, 0] + X[:, 1] > 1).astype(int)
t = time.time()
lgb.train(dict(objective="binary", device_type=sys.argv[1], verbose=-1), lgb.Dataset(X, y), 50)
print(sys.argv[1], "OK", round(time.time() - t, 1), "s")
'''
for dev in ('cpu', 'gpu', 'cuda'):
    r = subprocess.run(['python', '-c', gpu_test, dev], capture_output=True, text=True)
    err = r.stderr.strip().splitlines()
    print(r.stdout.strip() or f'{dev} FAILED: {err[-1] if err else "?"}')

In [ ]:
# 4. Helpers: run a pipeline step (streamed + logged with the commit), wipe resumable parts, show disk.
def step(*args, log=None):
    name = log or '_'.join(str(a).replace('.py', '') for a in args) + '.log'
    log = Path(os.environ['BER_WORK']) / 'logs' / name
    t = time.time()
    with open(log, 'a') as f:
        f.write(f'# commit {COMMIT}  {time.ctime()}  {args}\n')
        p = subprocess.Popen(['python', '-u', *map(str, args)], cwd=SRC, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
        for line in p.stdout:
            print(line, end=''); f.write(line); f.flush()
        rc = p.wait()
    print(f'--> exit {rc} in {time.time() - t:.0f}s')
    if rc:
        raise RuntimeError(f'{args} failed ({rc}); see {log}')

def fresh(*names):
    for n in names:
        shutil.rmtree(Path(os.environ['BER_WORK']) / n, ignore_errors=True)

def disk():
    subprocess.run('df -h /kaggle/working | tail -1; du -sh /kaggle/working/* 2>/dev/null', shell=True)

def seed_indic():
    shutil.copy(REPO / 'code/business_entity_resolution/artifacts/indic_dict.json', os.environ['BER_WORK'])

## Smoke test (~15 min): 2% train sample in a separate work folder

Uses `work_smoke/` so its part files can never leak into the real run. Checks the whole train path end to end.

In [ ]:
set_paths('/kaggle/working/work_smoke', '/kaggle/working/output_smoke')
seed_indic()
step('prep.py', 'train')
step('block.py', 'train', '--sample', 0.02)
step('embed.py', 'train', '--sample', 0.02)
step('features.py', 'train_cand_sample')
step('train.py', 0.02)

In [ ]:
# Reuse the smoke run's cleaned train parquet in the real work folder (prep output does not depend
# on the sample), then drop the smoke folder to free disk.
set_paths('/kaggle/working/work')
seed_indic()
for f in ('train_s1.parquet', 'train_s23.parquet', 'train_truth.parquet'):
    shutil.move(f'/kaggle/working/work_smoke/{f}', os.environ['BER_WORK'])
shutil.rmtree('/kaggle/working/work_smoke'); shutil.rmtree('/kaggle/working/output_smoke', ignore_errors=True)
disk()

## Full train (25% S1 sample)

In [ ]:
set_paths('/kaggle/working/work')
# os.environ['BER_LGB_DEVICE'] = 'gpu'   # only if cell 3 printed "gpu OK" and it beat cpu
if not Path(os.environ['BER_WORK'], 'train_s23.parquet').exists():
    seed_indic()
    step('prep.py', 'train')
step('block.py', 'train', '--sample', 0.25)
fresh('train_cand_sample_parts')          # merged copy already written
step('embed.py', 'train', '--sample', 0.25)   # GPU: e5 kNN candidates + emb features
step('features.py', 'train_cand_sample')  # parts are rebuilt automatically when candidates change
step('train.py', 0.25)
print(open(Path(os.environ['BER_WORK'], 'decision.json')).read())
disk()

## Test inference + validation

In [ ]:
step('prep.py', 'test')
step('block.py', 'test')
fresh('test_cand_parts')
step('embed.py', 'test')
disk()
step('features.py', 'test_cand')
step('predict.py')
step(VALIDATOR, '--matching', '/kaggle/working/output/matching_results.tsv',
     '--candidate', '/kaggle/working/output/candidate_pairs.tsv', '--test-dir', f'{DATA}/test',
     log='validate.log')

In [ ]:
# Keep the small, useful artifacts (tagged with the commit) and optionally free the big ones.
art = Path(f'/kaggle/working/artifacts_{COMMIT}')
art.mkdir(exist_ok=True)
for f in ('model.txt', 'decision.json', 'indic_dict.json'):
    shutil.copy(Path(os.environ['BER_WORK'], f), art)
shutil.copytree(Path(os.environ['BER_WORK'], 'logs'), art / 'logs', dirs_exist_ok=True)
# fresh('train_cand_sample_feat_parts', 'test_cand_feat_parts')   # uncomment when disk is tight
disk()

## Embedding probe (GPU, ~10 min)

Measures how many blocking misses the e5 nearest-neighbour candidates recover, on the first 300k
sampled S2/S3 records. Needs `train_cand_sample.parquet` from blocking. Writes nothing the full run reuses.

In [ ]:
exec(CELLS[0], globals())   # pull latest code
step('embed.py', 'train', '--sample', 0.25, '--probe', 300_000)